In [2]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# Create an API clients
from anthropic import Anthropic

client = Anthropic()
# NOTE: the Claude 5 family (claude-sonnet-5, claude-opus-4-8) removed the
# `temperature` parameter — it returns a 400. This notebook demonstrates
# temperature, so it pins to Sonnet 4.6, which still accepts it.
model = "claude-sonnet-4-6"

In [4]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return next(block.text for block in message.content if block.type == "text")

In [5]:
messages = []

add_user_message(messages,
    "Write a 1 sentence description of a fake database"
)

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)
for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_011Ccr5KneFBYM1pJrod6fLc', container=None, content=[], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=18, output_tokens=1, output_tokens_details=None, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='Here', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' is a one sentence description of a fake database:\n\n**"Nova', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='Base"** is a

In [9]:
messages = []

add_user_message(messages,
    "Write a 1 sentence description of a fake database"
)

# Use the high-level streaming helper (client.messages.stream), which exposes
# .text_stream. The low-level client.messages.create(..., stream=True) returns a
# raw event iterator with no .text_stream helper.
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
) as stream:
    for text in stream.text_stream:
        print(text, end="")

Here is a 1 sentence description of a fake database:

**FruityBase** is a fictional database containing 10,000 records of imaginary exotic fruits, including their made-up nutritional values, whimsical names, and the fantasy regions of the world where they supposedly grow.

In [10]:
messages = []

add_user_message(messages,
    "Write a 1 sentence description of a fake database"
)

# Use the high-level streaming helper (client.messages.stream), which exposes
# .text_stream. The low-level client.messages.create(..., stream=True) returns a
# raw event iterator with no .text_stream helper.
with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
) as stream:
    for text in stream.text_stream:
        # print(text, end="")
        pass

stream.get_final_message()

ParsedMessage(id='msg_011Ccr6G82Kb2bZVuDCxMxQx', container=None, content=[ParsedTextBlock(citations=None, text='Here is a one sentence description of a fake database:\n\n**"NovaBase is a fictional cloud-based database management system that uses quantum-encrypted neural indexing to store and retrieve data at speeds of up to 10 terabytes per millisecond across its network of imaginary distributed servers."**', type='text', parsed_output=None)], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=18, output_tokens=68, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

## Tip: streaming is now the *default*, not just a UX nicety

The cells above treat streaming as a way to show tokens as they arrive. That's still true, but on current models the bigger reason to stream is **timeout protection**:

- Non-streaming requests must finish within the SDK's ~10-minute HTTP timeout. For large outputs that's a real risk, so the **Python SDK will raise a `ValueError`** if it estimates a non-streaming request will exceed it (roughly any `max_tokens` above ~16K). Streaming sidesteps the guard.
- Current models (Sonnet 5, Opus 4.8, Fable 5) allow up to **128K output tokens** — which you can only realistically collect by streaming.

So the guidance flipped: **default to streaming for anything with long input, long output, or a high `max_tokens`**, and reach for non-streaming `client.messages.create(...)` only for short, bounded responses.

You don't lose the "just give me the finished message" ergonomics: prefer the high-level `client.messages.stream()` helper and call **`.get_final_message()`** — you get the complete `Message` object *and* timeout protection, with `.text_stream` still available if you also want live tokens.

> Bonus for thinking models: with adaptive thinking the default `display` is `"omitted"`, so a streaming UI shows a long pause before any text. Pass `thinking={"type": "adaptive", "display": "summarized"}` to stream the reasoning summary too.

The streaming cells here use no `temperature`, so — unlike the `chat()` helper — they run unchanged on a current model:

In [ ]:
# Recommended default for long / high-max_tokens output: stream, then collect the
# whole thing with .get_final_message(). This avoids the SDK's ~10-min HTTP timeout
# on large non-streaming requests (the Python SDK will even raise ValueError for a
# big non-streaming max_tokens). The streaming demos use no temperature, so they run
# unchanged on a current model.
messages = []
add_user_message(messages, "Write a 1 sentence description of a fake database")

with client.messages.stream(
    model="claude-sonnet-5",   # streaming needs no temperature -> a current model is fine
    max_tokens=64000,          # give long output room; streaming keeps it under the timeout
    messages=messages,
) as stream:
    for text in stream.text_stream:
        print(text, end="")

final = stream.get_final_message()   # full message object + timeout protection
print("\n\noutput tokens:", final.usage.output_tokens)